# [기초-실습] 통계 101×데이터 분석: (4장) 추론통계~신뢰구간

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

## ⚙️ 환경 준비 — 한글 폰트 설치 및 라이브러리 불러오기

In [ ]:
# 구글 코랩 환경에서 한글 폰트 설치 및 설정하기
# 필요시 아래 코드 실행 후, [런타임] - [세션 다시 시작] 후 셀을 다시 실행하세요.
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

In [ ]:
# 파이썬 라이브러리 및 모듈 가져오기
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'NanumGothic'  # 기본 폰트 설정
plt.rcParams['axes.unicode_minus'] = False   # 마이너스 기호 깨짐 방지

# 문제 1. 표본조사 체험하기

**📘 문제**

- 온라인 쇼핑몰은 전체 고객 수가 너무 많아, 모든 고객을 조사하기 어렵습니다.

- 그래서 무작위로 고객 30명을 뽑아 평균 만족도를 계산하고 이를 전체 만족도의 추정값으로 사용하려 합니다.

- 이번 실습에서는 직접 표본을 뽑고, 표본 평균을 구해보며,
  **“표본마다 결과가 달라질 수 있다”**는 추론 통계의 핵심 개념을 체험해봅니다.

In [ ]:
# 모집단 생성 (전체 고객 만족도 10,000명)
np.random.seed(2025)
population = np.random.normal(loc=7.5, scale=1.2, size=10000)
population = np.clip(population, 1, 10)  # 1점 ~ 10점 사이로 제한
df_pop = pd.DataFrame({'score': population})

# 전체 모집단 시각화
sns.histplot(df_pop['score'], bins=30, kde=True)
plt.title("전체 고객 만족도 분포 (모집단)")
plt.xlabel("만족도 점수")
plt.show()

**📌 아래를 수행해 보세요:**

- 표본을 무작위로 여러 번 뽑아 보고, 표본 평균이 어떻게 변하는지 확인해봅시다.

- 히스토그램을 그리고, 표본 평균의 분포 형태를 관찰해봅시다.

In [ ]:
# [문제 1] Q1. 모집단에서 무작위로 30명을 뽑아 표본 평균을 구해봅시다.
# 여기에 코드를 작성해주세요.
# [문제 1] Q1. 모집단에서 무작위로 30명을 뽑아 표본 평균을 구해봅시다.
sample = np.random.choice(population, size=30, replace=False)
sample_mean = sample.mean()

print("표본 평균:", sample_mean)


In [ ]:
# [문제 1] Q2. 이 과정을 500번 반복하고, 표본 평균을 리스트에 저장합니다.
# 여기에 코드를 작성해주세요.
# [문제 1] Q2. 이 과정을 500번 반복하고, 표본 평균을 리스트에 저장합니다.
sample_means = []

for i in range(500):
    sample = np.random.choice(population, size=30, replace=False)
    sample_means.append(sample.mean())

sample_means = np.array(sample_means)

print("표본 평균 리스트 개수:", len(sample_means))
print("표본 평균들의 평균:", sample_means.mean())


In [ ]:
# [문제 1] Q3. 표본 평균들의 분포를 히스토그램으로 그려봅시다. 평균선도 함께 표시해 봅시다.
# 여기에 코드를 작성해주세요.
# [문제 1] Q3. 표본 평균들의 분포를 히스토그램으로 그려봅시다. 평균선도 함께 표시해 봅시다.
sns.histplot(sample_means, bins=30, kde=True)
plt.axvline(sample_means.mean(), color='red', linestyle='--', label=f'평균: {sample_means.mean():.2f}')
plt.title("표본 평균들의 분포 (중심극한정리)")
plt.xlabel("표본 평균")
plt.ylabel("빈도")
plt.legend()
plt.show()

**🧠 데이터를 어떻게 읽을까요?**

- 표본 평균들은 어떤 값 주변에 많이 분포해 있나요? 이 값은 전체 모집단 평균과 얼마나 비슷한가요?

- 표본을 1번 뽑았을 때와 500번을 반복해서 뽑았을 때, 표본 평균의 분포나 신뢰성에는 어떤 차이가 있나요

- 친구가 다른 표본을 뽑았다면 같은 평균이 나왔을까요? 비슷한 결과가 나왔더라도 완전히 같지 않았다면, 그 이유는 무엇일까요?

- 표본 평균들의 분포는 어떤 모양인가요? 종 모양의 정규분포처럼 보이나요? 그렇다면 왜 그렇게 되는 걸까요?

In [ ]:
# [문제 1] 데이터를 어떻게 읽을까요?
# 여기에 의견을 작성해주세요.
1. 표본 평균들은 어떤 값 주변에 많이 분포해 있나요? 모집단 평균과 얼마나 비슷한가요?

표본 평균들은 모집단 평균인 7.5 근처에 가장 많이 몰려 있을 거예요. 실제로 sample_means.mean()을 계산해보면 7.5에 매우 가까운 값이 나올 텐데, 이건 우연이 아니라 통계학의 기본 원리 때문이에요. 표본 평균의 기댓값(평균의 평균)은 모집단 평균과 같아지거든요 (E[X̄] = μ). 표본을 500번이나 뽑아서 평균을 냈기 때문에, 개별 표본에서 생기는 오차들이 서로 상쇄되면서 진짜 모집단 평균에 아주 가까운 값으로 수렴한 거예요.

2. 1번 뽑았을 때 vs 500번 반복했을 때 - 차이는?

1번만 뽑으면: 그 표본 평균 하나만 얻게 되는데, 이 값이 우연히 모집단 평균보다 높거나 낮을 수 있어요. 즉, "이 표본 평균이 얼마나 믿을 만한지" 판단할 근거가 없어요.
500번 반복하면: 표본 평균들이 이루는 분포 전체를 볼 수 있어요. 이를 통해 표본 평균이 대체로 어디에 몰려있는지(중심 경향), 얼마나 퍼져있는지(변동성, 즉 표준오차)를 알 수 있죠. 즉 "표본 평균이라는 것 자체가 얼마나 안정적인 추정치인지"를 판단할 수 있게 되는 거예요. 이게 바로 표본 평균의 신뢰성을 정량적으로 평가할 수 있게 해주는 핵심 차이예요.

3. 친구가 다른 표본을 뽑았다면 같은 평균이 나왔을까요?

아마 완전히 같지는 않았을 거예요. 그 이유는 표본추출의 무작위성 때문이에요. 10,000명 중에서 30명을 뽑을 때마다 어떤 사람이 뽑히는지는 매번 달라지고, 뽑힌 사람들의 점수 조합도 달라지니까 평균도 조금씩 다르게 나오는 거죠. 이걸 **표본 변동(sampling variability)**이라고 불러요. 다만 완전히 다른 방향으로 튀지는 않을 거예요 — 30명 정도의 표본이면 대체로 모집단 평균 근처(7.5 근방)에서 크게 벗어나지 않는 값이 나올 가능성이 높아요.

4. 표본 평균들의 분포는 어떤 모양인가요? 정규분포처럼 보이나요?

네, 종 모양의 정규분포에 가깝게 보일 거예요. 이건 중심극한정리(Central Limit Theorem, CLT) 때문이에요. 중심극한정리에 따르면, 모집단의 분포 모양과 상관없이(원래 모집단이 정규분포가 아니어도!) 표본 크기가 충분히 크면(보통 n≥30 정도) 표본 평균들의 분포는 정규분포에 가까워져요.

이번 예제에서는 모집단 자체가 이미 정규분포(np.random.normal)를 따르기 때문에 더더욱 표본 평균 분포가 깔끔한 종 모양으로 나타날 거예요. 표본 크기가 30이라 CLT 조건도 충족하고요. 만약 표본 크기를 더 늘리면(예: 100명씩) 이 종 모양이 더 뾰족해지고 좁아지는 것도 확인할 수 있어요 — 표본이 커질수록 표본 평균의 변동성(표준오차)이 줄어들기 때문이에요.

# 문제 2. 중심극한정리

**📘 문제**

- 현실에서는 모집단의 분포가 정규분포가 아닐 수도 있습니다.

- 예를 들어, 일부 고객은 매우 높은 점수를 주고, 대부분은 낮은 점수를 주는 만족도 분포가 있을 수 있죠. (예: 지수분포)

- 이처럼 원래 분포가 비정규분포여도,
  표본을 여러 번 뽑아 평균을 계산하면, 그 평균들의 분포는 정규분포에 가까워진다는 것을
  **중심극한정리(Central Limit Theorem)**라고 합니다.

- 이번 실습에서는 다양한 크기의 표본을 뽑아 평균을 계산하고,
  그 평균들의 분포가 어떻게 변하는지를 직접 실험해 봅니다.

In [ ]:
# 지수분포를 따르는 모집단 생성
np.random.seed(2025)
population = np.random.exponential(scale=50, size=100000)  # 평균 50, 비대칭 분포

# 모집단 시각화
sns.histplot(population, bins=50, kde=True)
plt.title("고객 구매 금액 분포 (모집단: 지수분포)")
plt.xlabel("구매 금액")
plt.show()

**📌 아래를 수행해 보세요:**

- 비대칭적인 모집단(지수분포)에서 무작위로 표본을 추출해 평균을 구해봅시다.

- 표본 크기를 바꿔가며, 표본 평균들의 분포가 어떻게 변화하는지 확인해봅시다.

- 히스토그램을 그리고, 분포의 모양을 관찰해봅시다.

- 표본 크기가 커질수록 표본 평균 분포의 모양과 **퍼진 정도(분산)**가 어떻게 변하는지 관찰해봅시다.

In [ ]:
# [문제 2] Q1. 모집단에서 표본을 1000번 뽑고, 각 표본의 평균을 구해봅시다.
# 표본 크기 = 5일 때

# 여기에 코드를 작성해주세요.

sample_means_n5 = []

for i in range(1000):
    sample = np.random.choice(population, size=5, replace=False)
    sample_means_n5.append(sample.mean())

sample_means_n5 = np.array(sample_means_n5)

print("표본 크기 5, 표본 평균 개수:", len(sample_means_n5))
print("표본 평균들의 평균:", sample_means_n5.mean())
print("표본 평균들의 표준편차:", sample_means_n5.std())

In [ ]:
# [문제 2] Q2. 위 과정을 표본 크기 30, 100일 때도 반복해봅시다.
# sample_size = 30, 100

# 여기에 코드를 작성해주세요.
# [문제 2] Q2. 위 과정을 표본 크기 30, 100일 때도 반복해봅시다.

# 표본 크기 = 30일 때
sample_means_n30 = []

for i in range(1000):
    sample = np.random.choice(population, size=30, replace=False)
    sample_means_n30.append(sample.mean())

sample_means_n30 = np.array(sample_means_n30)

print("표본 크기 30, 표본 평균들의 평균:", sample_means_n30.mean())
print("표본 크기 30, 표본 평균들의 표준편차:", sample_means_n30.std())

# 표본 크기 = 100일 때
sample_means_n100 = []

for i in range(1000):
    sample = np.random.choice(population, size=100, replace=False)
    sample_means_n100.append(sample.mean())

sample_means_n100 = np.array(sample_means_n100)

print("표본 크기 100, 표본 평균들의 평균:", sample_means_n100.mean())
print("표본 크기 100, 표본 평균들의 표준편차:", sample_means_n100.std())

In [ ]:
# [문제 2] Q3. 각 표본 크기별로 표본 평균들의 분포를 히스토그램으로 그려봅시다.
# 평균선을 함께 표시해 봅시다.

# 여기에 코드를 작성해주세요.
# [문제 2] Q3. 각 표본 크기별로 표본 평균들의 분포를 히스토그램으로 그려봅시다.
# 평균선을 함께 표시해 봅시다.

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 표본 크기 = 5
sns.histplot(sample_means_n5, bins=30, kde=True, ax=axes[0])
axes[0].axvline(sample_means_n5.mean(), color='red', linestyle='--', 
                label=f'평균: {sample_means_n5.mean():.2f}')
axes[0].set_title("표본 크기 5")
axes[0].set_xlabel("표본 평균")
axes[0].set_ylabel("빈도")
axes[0].legend()

# 표본 크기 = 30
sns.histplot(sample_means_n30, bins=30, kde=True, ax=axes[1])
axes[1].axvline(sample_means_n30.mean(), color='red', linestyle='--', 
                label=f'평균: {sample_means_n30.mean():.2f}')
axes[1].set_title("표본 크기 30")
axes[1].set_xlabel("표본 평균")
axes[1].set_ylabel("빈도")
axes[1].legend()

# 표본 크기 = 100
sns.histplot(sample_means_n100, bins=30, kde=True, ax=axes[2])
axes[2].axvline(sample_means_n100.mean(), color='red', linestyle='--', 
                label=f'평균: {sample_means_n100.mean():.2f}')
axes[2].set_title("표본 크기 100")
axes[2].set_xlabel("표본 평균")
axes[2].set_ylabel("빈도")
axes[2].legend()

plt.suptitle("표본 크기별 표본 평균 분포 비교 (중심극한정리)")
plt.tight_layout()
plt.show()

**🧠 데이터를 어떻게 읽을까요?**

- 표본 크기가 작을 때 (예: 5), 평균들의 분포는 어떤 모양인가요?

- 표본 크기가 커질수록 평균 분포의 모양은 어떤 변화를 보이나요?

- 원래 모집단은 비대칭이었는데, 왜 평균들의 분포는 정규분포처럼 바뀌었을까요?

- 이 실험을 통해 중심극한정리를 어떻게 이해하게 되었나요?

- 표본 크기에 따라 **분포의 넓이(흩어짐)**는 어떻게 달라지나요?

In [ ]:
# [문제 2]데이터를 어떻게 읽을까요?
# 여기에 의견을 작성해주세요.
1. 표본 크기가 작을 때(예: 5), 평균들의 분포는 어떤 모양인가요?

원래 모집단인 지수분포의 비대칭적인 특성이 어느 정도 남아있는 모양일 거예요. 왼쪽으로 치우치고 오른쪽으로 긴 꼬리를 가진(오른쪽 비대칭, right-skewed) 형태가 보일 텐데, 이는 표본 크기 5는 지수분포 같은 강한 비대칭 모집단의 "비정상성"을 완전히 상쇄하기엔 너무 작기 때문이에요. 예를 들어 5명 중 우연히 구매 금액이 아주 큰 사람(예: 200 이상)이 하나만 포함돼도 그 표본 평균은 확 튀어버리거든요. 그래서 표본 평균들의 분포에도 이런 극단값의 영향이 꽤 남아있어요.

2. 표본 크기가 커질수록 평균 분포의 모양은 어떤 변화를 보이나요?

n=5 → n=30 → n=100으로 갈수록 분포가 점점 좌우 대칭에 가까워지고, 종 모양(정규분포)의 형태로 바뀌어요. 비대칭성(왜도)이 점점 줄어들면서 매끈한 정규분포 곡선에 가까워지는 걸 히스토그램에서 눈으로 확인할 수 있을 거예요.

3. 원래 모집단은 비대칭이었는데, 왜 평균들의 분포는 정규분포처럼 바뀌었을까요?

이게 바로 **중심극한정리(CLT)**의 핵심이에요. 표본 평균은 여러 개의 개별 관측값을 더해서 나눈 값이죠. 극단적으로 크거나 작은 값 하나가 전체 평균에 미치는 영향은, 표본 크기가 커질수록 "희석"돼요. 즉 n개의 값을 더하면 우연히 큰 값과 작은 값들이 서로 상쇄되는 효과가 커지기 때문에, 개별 데이터의 비대칭성이 평균이라는 연산을 거치면서 점점 상쇄되고 사라지는 거예요. 이 상쇄 효과가 표본 크기가 커질수록 강해지기 때문에 모집단 모양과 무관하게 표본 평균의 분포는 정규분포에 수렴하게 됩니다.

4. 이 실험을 통해 중심극한정리를 어떻게 이해하게 되었나요?

이 실험이 보여주는 건, 중심극한정리가 단순히 "정규분포를 가정해도 된다"는 이론적 약속이 아니라, 실제로 표본을 반복해서 뽑고 평균을 내는 과정에서 눈으로 확인할 수 있는 현상이라는 점이에요. 특히 원래 모집단이 정규분포와 전혀 다른 형태(지수분포처럼 심하게 비대칭인 경우)여도, 표본 크기만 충분히 크면(대략 n≥30) 표본 평균의 분포는 정규분포로 근사된다는 걸 실제 시뮬레이션으로 검증한 셈이죠. 이 덕분에 실무에서 모집단 분포를 몰라도 표본 평균에 대해 정규분포 기반의 통계적 추론(신뢰구간, 가설검정 등)을 적용할 수 있다는 걸 체감할 수 있어요.

5. 표본 크기에 따라 분포의 넓이(흩어짐)는 어떻게 달라지나요?

표본 크기가 커질수록 표본 평균들의 분포는 점점 더 좁아져요 (분산이 줄어듦). 이는 표준오차(SE) 공식인 SE = σ/√n 때문이에요. n이 커질수록 분모인 √n이 커지므로 SE는 작아지죠. 실제로 위 코드에서 계산한 sample_means_n5.std(), sample_means_n30.std(), sample_means_n100.std() 값을 비교해보면, n=5일 때 표준편차가 가장 크고, n=100일 때 가장 작을 거예요. 이는 표본 크기가 클수록 표본 평균이 모집단 평균을 더 정밀하게 추정한다는 걸 의미해요 — 표본이 클수록 "운 나쁘게" 극단적인 표본 평균이 나올 가능성이 줄어드는 거죠.

# 문제 3. 표준오차

**📘 문제**

- 앞선 실습에서 우리는 **표본 크기(n)가 커질수록 표본 평균들의 분포가 더 좁아진다**는 것을 확인했습니다.
- 이처럼 표본 평균들이 얼마나 흩어져 있는지(분포의 퍼진 정도)를 나타내는 값을 **표준오차(Standard Error, SE)**라고 부릅니다.
- 표준오차는 **표본 평균들의 표준편차**와 같은 의미이며, 이는 우리가 뽑은 표본 평균이 실제 모평균과 평균적으로 얼마나 떨어져 있을지를 나타내는 **'예상 오차의 크기'**입니다.

- 통계학적으로 이 표준오차는 **`SE = σ / √n`** (모집단 표준편차 / 표본 크기의 제곱근) 이라는 공식으로 계산할 수 있습니다.
- 이 공식은 **표본 크기(n)가 커질수록 표준오차(SE)가 작아진다**는 것을 명확히 보여줍니다.

- 이번 실습에서는 여러 크기의 표본을 뽑아, 시뮬레이션을 통해 얻은 **표본 평균들의 표준편차(실험값)**가 공식으로 계산한 **표준오차(이론값)**와 얼마나 일치하는지 직접 확인해봅니다.

In [ ]:
# 모집단 생성 (평균 100, 표준편차 15)
np.random.seed(2025)
population = np.random.normal(loc=100, scale=15, size=100000)

# 모집단 시각화
sns.histplot(population, bins=40, kde=True)
plt.title("모집단 분포 (평균 100, 표준편차 15)")
plt.xlabel("값")
plt.show()

**📌 아래를 수행해 보세요:**

- 모집단에서 여러 크기의 표본(10, 30, 100, 500)을 각각 1000번 뽑고, 그 평균들을 구한 뒤, **표본 평균들의 표준편차(=실험적 표준오차)**를 계산해봅시다.

- 이 결과를 이론적인 표준오차 공식과 비교하는 표를 만들고, 시각화해봅시다.

In [ ]:
# [문제 3] Q1. 표본 크기 10, 30, 100, 500에 대해 각각 1000번 표본을 뽑고, 평균을 구해봅시다.
# 각 표본 평균 분포의 표준편차를 계산해봅시다.
# 결과를 리스트에 저장하고, 표로 정리해봅시다.

# 여기에 코드를 작성해주세요.



sample_sizes = [10, 30, 100, 500]
results = []
sample_means_dict = {}  # 각 표본 크기별 표본 평균 리스트를 저장 (이후 시각화용)

for n in sample_sizes:
    means = []
    for i in range(1000):
        sample = np.random.choice(population, size=n, replace=False)
        means.append(sample.mean())
    means = np.array(means)
    
    sample_means_dict[n] = means
    
    results.append({
        '표본 크기': n,
        '표본 평균들의 평균': means.mean(),
        '표본 평균들의 표준편차 (실제)': means.std(),
        '이론적 표준오차 (σ/√n)': 15 / np.sqrt(n)
    })

df_results = pd.DataFrame(results)
df_results

In [ ]:
# [문제 3] Q2. 이론적인 표준오차와 비교해봅시다.
# [공식] 표준오차(SE) = 모집단 표준편차 / √표본크기

# 여기에 코드를 작성해주세요.


population_std = population.std()  # 모집단의 실제 표준편차 (약 15에 근접)
print("모집단 표준편차:", population_std)
print()

comparison = []

for n in sample_sizes:
    actual_std = sample_means_dict[n].std()
    theoretical_se = population_std / np.sqrt(n)
    diff = actual_std - theoretical_se
    
    comparison.append({
        '표본 크기': n,
        '실제 표본평균 표준편차': round(actual_std, 4),
        '이론적 표준오차 (σ/√n)': round(theoretical_se, 4),
        '차이': round(diff, 4)
    })

df_comparison = pd.DataFrame(comparison)
df_comparison

In [ ]:
# [문제 3] Q3. 실험값과 이론값을 시각화해봅시다.
# 표본 크기를 x축, 표준오차를 y축으로 한 꺾은선 그래프를 그려봅시다.

# 여기에 코드를 작성해주세요.
# [문제 3] Q3. 실험값과 이론값을 시각화해봅시다.
# 표본 크기를 x축, 표준오차를 y축으로 한 꺾은선 그래프를 그려봅시다.

actual_stds = [sample_means_dict[n].std() for n in sample_sizes]
theoretical_ses = [population_std / np.sqrt(n) for n in sample_sizes]

plt.figure(figsize=(8, 6))
plt.plot(sample_sizes, actual_stds, marker='o', label='실제 표본평균 표준편차', color='blue')
plt.plot(sample_sizes, theoretical_ses, marker='s', linestyle='--', label='이론적 표준오차 (σ/√n)', color='red')

plt.title("표본 크기에 따른 표준오차 변화 (실험값 vs 이론값)")
plt.xlabel("표본 크기 (n)")
plt.ylabel("표준오차 (Standard Error)")
plt.legend()
plt.grid(True)
plt.show()

**🧠 데이터를 어떻게 읽을까요?**

- 표본 크기가 작을수록, 표본 평균의 분포는 어떤 모양인가요? 넓게 퍼져 있나요?

- 표본 크기가 커질수록, 평균 분포는 어떻게 변하나요?

- 실험값과 이론값(공식 계산값)은 얼마나 비슷한가요?

- 왜 표본 크기가 커질수록 표준오차는 작아질까요?

In [ ]:
# [문제 3] 데이터를 어떻게 읽을까요?
# 여기에 의견을 작성해주세요.
1. 표본 크기가 작을수록, 표본 평균의 분포는 어떤 모양인가요? 넓게 퍼져 있나요?

네, 표본 크기가 작을수록(n=10) 표본 평균들의 분포는 훨씬 넓게 퍼져 있어요. 이번 실험의 모집단은 원래 정규분포라서 모양 자체는 n=10에서도 이미 종 모양(정규분포)으로 나타나겠지만, 그 폭이 넓어서 표본 평균값들이 100에서 꽤 멀리 떨어진 값들도(예: 90, 110 근처) 자주 나타날 거예요. 표본이 10명밖에 안 되니까, 우연히 평균보다 높거나 낮은 사람들이 몰려서 뽑히면 그 영향이 표본 평균에 고스란히 반영되는 거죠.

2. 표본 크기가 커질수록, 평균 분포는 어떻게 변하나요?

n=10 → 30 → 100 → 500으로 갈수록 분포는 점점 좁아지고 100 주변으로 뾰족하게 몰려요. 모양 자체는 계속 종 모양(정규분포)을 유지하지만, 폭(퍼짐 정도)만 계속 줄어드는 거예요. n=500쯤 되면 표본 평균들이 거의 대부분 99~101 사이의 아주 좁은 범위에 몰려있는 걸 확인할 수 있을 거예요.

3. 실험값과 이론값(공식 계산값)은 얼마나 비슷한가요?

Q2에서 만든 df_comparison 표를 보면, 실제 시뮬레이션으로 얻은 표준편차와 이론적 표준오차(σ/√n)의 차이가 소수점 둘째~셋째 자리 수준으로 거의 무시할 만큼 작을 거예요. 이는 1000번이라는 충분히 많은 반복 횟수 덕분에 시뮬레이션 결과가 이론값에 아주 가깝게 수렴했기 때문이에요. 즉, "표본 평균의 표준편차는 σ/√n을 따른다"는 통계 이론이 실제 데이터로도 정확히 검증된 셈이죠.

4. 왜 표본 크기가 커질수록 표준오차는 작아질까요?

수식으로 보면 SE = σ/√n이니, n이 커지면 분모가 커져서 SE는 자연히 작아져요. 그런데 이걸 좀 더 직관적으로 이해하면:

표본 크기가 커진다는 건 평균을 계산할 때 더 많은 값들을 함께 평균 낸다는 뜻이에요. 개별 관측값 하나하나는 우연에 따라 크거나 작게 나올 수 있지만, 표본에 포함된 값의 개수가 많아질수록 우연히 큰 값과 작은 값들이 서로 상쇄되는 효과가 강해져요. 즉, "운 좋게 극단적으로 높은/낮은 사람들만 뽑힐 확률"이 표본 크기가 커질수록 급격히 줄어드는 거죠. 그래서 표본 평균은 점점 더 안정적으로 모집단 평균(100) 근처의 값으로 수렴하게 되고, 그 결과 표본 평균들 사이의 흩어짐(표준오차)도 작아지는 거예요.

다만 √n 관계라서 감소 속도가 일정하지 않다는 것도 흥미로운 포인트예요 — 표본 크기를 4배로 늘려야 표준오차가 절반(1/2)으로 줄어들거든요. 그래서 n=10→30처럼 초반에는 표준오차가 빠르게 줄어들지만, n=100→500처럼 이미 표본이 큰 상태에서는 표준오차가 줄어드는 속도가 상대적으로 완만해지는 것도 그래프에서 확인할 수 있어요.

# 문제 4. 신뢰구간 계산과 해석

**📘 문제**

- 표본 평균은 모집단 평균을 추정하는 좋은 점 추정(Point Estimation) 값이지만, 표본오차 때문에 정확히 일치하지는 않습니다.

- 그래서 우리는 "모집단 평균이 아마 이 범위 안에 있을 것이다"라고 **구간으로 추정(Interval Estimation)**하는 것이 더 합리적입니다. 이때 사용하는 개념이 바로 **신뢰구간(Confidence Interval)**입니다.

- 신뢰구간은 표본평균 ± 오차범위 형태로 계산되며, 이 오차범위는 신뢰수준(예: 95%, 99%)과 표본오차에 의해 결정됩니다.

- 이번 실습에서는 **모집단 표준편차(σ)를 알 때(z-분포)**와 **모를 때(t-분포)**의 신뢰구간을 각각 계산해보고, 신뢰수준에 따라 구간의 폭이 어떻게 변하는지 확인해봅니다.

In [ ]:
# 모집단 생성
np.random.seed(2025)
population = np.random.normal(loc=70, scale=10, size=10000)

# 모집단 시각화
sns.histplot(population, bins=40, kde=True)
plt.title("모집단 분포 (평균 70, 표준편차 10)")
plt.xlabel("점수")
plt.show()

**📌 아래를 수행해 보세요:**

- 모집단에서 30명을 무작위로 뽑아 평균, 표준편차, 표준오차를 계산해보세요.

- 95% 신뢰구간을 z-분포와 t-분포를 각각 사용해서 계산해보세요.

- 신뢰수준을 바꿨을 때(90%, 99%) 신뢰구간이 어떻게 변하는지 확인해보세요.

In [ ]:
# [문제 4] Q1. 모집단에서 표본 30명을 무작위로 추출하고, 표본 평균과 표준편차를 구해봅시다.
# 표준오차도 함께 계산해보세요.

# 여기에 코드를 작성해주세요.
# [문제 4] Q1. 모집단에서 표본 30명을 무작위로 추출하고, 표본 평균과 표준편차를 구해봅시다.
# 표준오차도 함께 계산해보세요.

np.random.seed(42)  # 재현성을 위한 시드 설정 (원하는 값으로 변경 가능)

sample = np.random.choice(population, size=30, replace=False)

sample_mean = sample.mean()
sample_std = sample.std(ddof=1)  # 표본표준편차 (n-1로 나눔)
n = len(sample)
standard_error = sample_std / np.sqrt(n)

print("표본 평균:", sample_mean)
print("표본 표준편차:", sample_std)
print("표준오차 (SE):", standard_error)

In [ ]:
# [문제 4] Q2. 모집단의 표준편차를 알고 있다고 가정하고, z-분포를 사용하여 95% 신뢰구간을 계산해봅시다.

# 여기에 코드를 작성해주세요.
# [문제 4] Q2. 모집단의 표준편차를 알고 있다고 가정하고, z-분포를 사용하여 95% 신뢰구간을 계산해봅시다.

from scipy import stats

population_std_known = 10  # 모집단 표준편차를 알고 있다고 가정 (population 생성 시 scale=10)

# 95% 신뢰수준에 해당하는 z값
confidence_level = 0.95
z_critical = stats.norm.ppf(1 - (1 - confidence_level) / 2)

# 표준오차 (모집단 표준편차 기준)
se_z = population_std_known / np.sqrt(n)

# 신뢰구간 계산
margin_of_error_z = z_critical * se_z
ci_lower_z = sample_mean - margin_of_error_z
ci_upper_z = sample_mean + margin_of_error_z

print("z-critical 값:", z_critical)
print("표준오차 (SE, 모집단 표준편차 기준):", se_z)
print(f"95% 신뢰구간 (z-분포): ({ci_lower_z:.4f}, {ci_upper_z:.4f})")

In [ ]:
# [문제 4] Q3. 모집단의 표준편차를 모른다고 가정하고, 표본 표준편차와 t-분포를 사용하여 95% 신뢰구간을 계산해봅시다.

# 여기에 코드를 작성해주세요.
# [문제 4] Q3. 모집단의 표준편차를 모른다고 가정하고, 표본 표준편차와 t-분포를 사용하여 95% 신뢰구간을 계산해봅시다.

# 자유도 (degree of freedom)
df = n - 1

# 95% 신뢰수준에 해당하는 t값
t_critical = stats.t.ppf(1 - (1 - confidence_level) / 2, df)

# 표준오차 (표본 표준편차 기준, Q1에서 계산한 standard_error 사용)
se_t = standard_error  # sample_std / sqrt(n)

# 신뢰구간 계산
margin_of_error_t = t_critical * se_t
ci_lower_t = sample_mean - margin_of_error_t
ci_upper_t = sample_mean + margin_of_error_t

print("자유도 (df):", df)
print("t-critical 값:", t_critical)
print("표준오차 (SE, 표본 표준편차 기준):", se_t)
print(f"95% 신뢰구간 (t-분포): ({ci_lower_t:.4f}, {ci_upper_t:.4f})")

In [ ]:
# [문제 4] Q4. 신뢰수준을 90%, 99%로 바꿔가며 신뢰구간을 계산해보고, 그 폭을 비교해봅시다.

# 여기에 코드를 작성해주세요.
# [문제 4] Q4. 신뢰수준을 90%, 99%로 바꿔가며 신뢰구간을 계산해보고, 그 폭을 비교해봅시다.

confidence_levels = [0.90, 0.95, 0.99]
ci_results = []

for cl in confidence_levels:
    # t-분포 기준 (표본 표준편차 사용)
    t_crit = stats.t.ppf(1 - (1 - cl) / 2, df)
    margin = t_crit * se_t
    lower = sample_mean - margin
    upper = sample_mean + margin
    width = upper - lower
    
    ci_results.append({
        '신뢰수준': f"{int(cl*100)}%",
        't-critical': round(t_crit, 4),
        '오차범위 (margin of error)': round(margin, 4),
        '신뢰구간 하한': round(lower, 4),
        '신뢰구간 상한': round(upper, 4),
        '신뢰구간 폭': round(width, 4)
    })

df_ci = pd.DataFrame(ci_results)
print(df_ci)
print()

# 신뢰구간 폭 비교 시각화
plt.figure(figsize=(8, 6))

colors = ['green', 'blue', 'red']
for i, row in df_ci.iterrows():
    cl = row['신뢰수준']
    lower = row['신뢰구간 하한']
    upper = row['신뢰구간 상한']
    plt.plot([lower, upper], [i, i], marker='o', linewidth=3, color=colors[i], label=f"{cl} 신뢰구간")
    plt.text(upper + 0.1, i, f"폭: {row['신뢰구간 폭']:.2f}", va='center')

plt.axvline(sample_mean, color='black', linestyle='--', label='표본 평균')
plt.yticks(range(len(df_ci)), df_ci['신뢰수준'])
plt.xlabel("점수")
plt.title("신뢰수준별 신뢰구간 비교")
plt.legend()
plt

**🧠 데이터를 어떻게 읽을까요?**

- z-분포와 t-분포를 사용한 신뢰구간은 얼마나 차이가 있나요?

- 신뢰수준이 높아질수록 신뢰구간의 폭은 어떻게 변하나요? 왜 그럴까요?

- 신뢰구간이 넓다는 건 좋은 걸까요? 나쁜 걸까요?

- 이 데이터가 실제 고객 만족도라면, 신뢰구간 정보를 마케팅 전략에 어떻게 활용할 수 있을까요?

In [ ]:
# [문제 4] 데이터를 어떻게 읽을까요?
# 여기에 의견을 작성해주세요.
1. z-분포와 t-분포를 사용한 신뢰구간은 얼마나 차이가 있나요?

Q2(z-분포)와 Q3(t-분포)에서 계산한 결과를 비교해보면, t-critical 값(약 2.045, 자유도 29)이 z-critical 값(약 1.96)보다 살짝 커요. 그래서 t-분포로 만든 신뢰구간이 z-분포로 만든 신뢰구간보다 폭이 조금 더 넓게 나와요. 이유는 t-분포가 표본 표준편차를 사용하기 때문이에요 — 표본 표준편차는 모집단 표준편차의 추정치일 뿐이라 그 자체에 불확실성이 존재하고, t-분포는 이 추가적인 불확실성을 반영하기 위해 정규분포보다 꼬리가 더 두꺼운 형태를 갖고 있거든요. 다만 표본 크기 30 정도면 그 차이가 아주 크지는 않고, 표본 크기가 더 작아질수록(예: n=5, 10) 그 차이가 훨씬 뚜렷해져요.

2. 신뢰수준이 높아질수록 신뢰구간의 폭은 어떻게 변하나요? 왜 그럴까요?

Q4 결과를 보면 90% → 95% → 99%로 갈수록 신뢰구간의 폭이 점점 넓어져요. 이건 신뢰수준이 "이 구간 안에 진짜 모집단 평균이 있을 확률"을 의미하기 때문이에요. 더 높은 확률로 모수를 포함하고 싶다면, 그만큼 구간을 더 넓게 잡아서 "안전 마진"을 늘려야 해요. 반대로 구간을 좁게 잡으면 더 정밀해 보이지만, 그만큼 진짜 값을 놓칠 위험도 커지는 거죠. 즉 신뢰수준과 신뢰구간의 폭은 트레이드오프 관계에 있어요.

3. 신뢰구간이 넓다는 건 좋은 걸까요? 나쁜 걸까요?

한마디로 말하기는 어렵고, 상황에 따라 다르게 해석해야 해요.

좋은 점: 구간이 넓다는 건 그만큼 "확실성(신뢰수준)"을 높였다는 뜻일 수 있어요 (예: 99% 신뢰구간). 확실성이 중요한 상황(예: 의료 판단)에서는 넓더라도 신뢰수준이 높은 구간이 유리할 수 있어요.
나쁜 점: 하지만 실무적으로는 구간이 너무 넓으면 "그래서 평균이 정확히 어느 정도인지" 판단하기가 어려워져요. 예를 들어 "고객 만족도가 65~75점 사이"라는 신뢰구간은 너무 넓어서 의사결정에 별 도움이 안 되죠. 신뢰구간이 넓어지는 원인은 보통 표본 크기가 작거나(SE가 커짐), 데이터의 변동성(표준편차)이 크기 때문이에요. 그래서 신뢰구간을 좁히고 싶다면 표본 크기를 늘리는 것이 가장 현실적인 방법이에요 (SE = σ/√n 이므로 n을 늘리면 구간이 좁아짐).

즉, 신뢰구간의 "적절한 폭"이란 것은 신뢰수준과 정밀도 사이에서 목적에 맞게 균형을 잡는 것이라고 볼 수 있어요.

4. 이 데이터가 실제 고객 만족도라면, 신뢰구간 정보를 마케팅 전략에 어떻게 활용할 수 있을까요?

몇 가지 활용 방향을 생각해볼 수 있어요:

의사결정의 근거로 활용: 단순히 "표본 평균이 72점이다"라고 말하는 것보다, "95% 신뢰구간으로 68~76점 사이"라고 말하면 이 수치가 얼마나 신뢰할 만한지, 오차 범위가 어느 정도인지까지 함께 전달할 수 있어요. 예를 들어 신뢰구간의 하한선이 경쟁사보다 높다면 "우리 만족도가 확실히 더 낫다"고 자신 있게 말할 수 있지만, 신뢰구간이 겹친다면 "통계적으로 유의미한 차이라고 보기 어렵다"고 신중하게 접근해야 해요.
표본 크기 계획에 활용: 마케팅팀에서 "만족도를 ±2점 오차 범위 안에서 알고 싶다"는 목표가 있다면, 이 신뢰구간 공식을 거꾸로 이용해서 "그러려면 최소 몇 명을 조사해야 하는지" 표본 크기를 미리 설계할 수 있어요.
시계열 모니터링: 매달 신뢰구간을 계산해서 추이를 보면, 단순 평균값의 오르내림이 실제 유의미한 변화인지, 아니면 표본추출 과정에서 생기는 자연스러운 변동(노이즈)인지 구분할 수 있어요. 신뢰구간이 겹치지 않을 정도로 평균이 변했다면 마케팅 캠페인 등의 효과가 통계적으로 유의미하다고 판단할 근거가 되는 거죠.
리스크 관리: 신뢰구간의 하한값을 "최악의 시나리오"로 보고 보수적으로 의사결정을 내리는 데 활용할 수도 있어요. 예를 들어 신제품 만족도의 신뢰구간 하한이 여전히 목표치(예: 70점) 이상이라면, 표본에서 우연히 낮게 나왔더라도 실제로는 안정적으로 목표를 달성했다고 어느 정도 확신할 수 있어요.

# 문제 5. 미니 프로젝트 - 고객 만족도 신뢰구간 추정

**📘 문제**

- 전체 고객 10,000명을 대상으로 만족도 조사를 하는 것은 시간과 비용이 많이 듭니다.
- 그래서 우리는 무작위로 일부 고객만 조사하여, 전체 고객의 평균 만족도를 추정하려 합니다.

- 이 프로젝트에서는 실제와 같은 상황을 가정하여, 표본을 뽑고 신뢰구간을 계산한 뒤, 이 결과를 바탕으로 마케팅 전략에 어떻게 활용할 수 있을지까지 생각해보는 실습을 진행합니다.

In [ ]:
# 모집단 생성 (고객 만족도 10,000명)
np.random.seed(2025)
population = np.random.normal(loc=7.2, scale=1.0, size=10000)
population = np.clip(population, 1, 10)

# 모집단 시각화
sns.histplot(population, bins=30, kde=True)
plt.title("전체 고객 만족도 분포 (모집단)")
plt.xlabel("만족도 점수")
plt.show()

**📌 아래를 수행해 보세요:**

- 모집단을 생성하고, 거기서 표본을 40명 뽑아 평균을 계산해봅시다.

- 표본 평균과 표준편차를 바탕으로 95% 신뢰구간을 계산해봅시다.

- 히스토그램을 그리고 신뢰구간을 시각화해봅시다.

- 이 결과를 어떻게 해석하고, 마케팅 전략에 활용할 수 있을지 생각해봅시다.

In [ ]:
# [문제 5] Q1. 모집단에서 표본 40명을 무작위로 추출하고, 표본 평균과 표준편차를 구해봅시다.
# 여기에 코드를 작성해주세요.
# [문제 5] Q1. 모집단에서 표본 40명을 무작위로 추출하고, 표본 평균과 표준편차를 구해봅시다.

np.random.seed(42)  # 재현성을 위한 시드 설정

sample = np.random.choice(population, size=40, replace=False)

sample_mean = sample.mean()
sample_std = sample.std(ddof=1)  # 표본표준편차 (n-1로 나눔)
n = len(sample)

print("표본 크기:", n)
print("표본 평균:", sample_mean)
print("표본 표준편차:", sample_std)

In [ ]:
# [문제 5] Q2. 표준오차(SE)를 구하고, t-분포를 사용하여 95% 신뢰구간을 계산해봅시다.
# 여기에 코드를 작성해주세요.
# [문제 5] Q2. 표준오차(SE)를 구하고, t-분포를 사용하여 95% 신뢰구간을 계산해봅시다.

from scipy import stats

# 표준오차 계산
standard_error = sample_std / np.sqrt(n)

# 자유도
df = n - 1

# 95% 신뢰수준에 해당하는 t값
confidence_level = 0.95
t_critical = stats.t.ppf(1 - (1 - confidence_level) / 2, df)

# 신뢰구간 계산
margin_of_error = t_critical * standard_error
ci_lower = sample_mean - margin_of_error
ci_upper = sample_mean + margin_of_error

print("표준오차 (SE):", standard_error)
print("자유도 (df):", df)
print("t-critical 값:", t_critical)
print(f"95% 신뢰구간: ({ci_lower:.4f}, {ci_upper:.4f})")

In [ ]:
# [문제 5] Q3. 표본 데이터의 히스토그램을 그리고, 평균 및 신뢰구간을 함께 시각화해봅시다.
#  여기에 코드를 작성해주세요.
# [문제 5] Q3. 표본 데이터의 히스토그램을 그리고, 평균 및 신뢰구간을 함께 시각화해봅시다.

plt.figure(figsize=(9, 6))
sns.histplot(sample, bins=15, kde=True, color='skyblue')

plt.axvline(sample_mean, color='red', linestyle='-', linewidth=2, label=f'표본 평균: {sample_mean:.2f}')
plt.axvline(ci_lower, color='green', linestyle='--', linewidth=2, label=f'신뢰구간 하한: {ci_lower:.2f}')
plt.axvline(ci_upper, color='green', linestyle='--', linewidth=2, label=f'신뢰구간 상한: {ci_upper:.2f}')

# 신뢰구간 영역을 색으로 표시
plt.axvspan(ci_lower, ci_upper, color='green', alpha=0.1, label='95% 신뢰구간')

plt.title("표본 데이터 분포와 95% 신뢰구간")
plt.xlabel("만족도 점수")
plt.ylabel("빈도")
plt.legend()
plt.show()

In [ ]:
# [문제 5] Q4. 신뢰구간의 결과에 따라 어떤 구체적인 마케팅 전략을 세울 수 있을까요?
# 여기에 의견을 작성해주세요.
신뢰구간 결과 해석 및 마케팅 전략 제안

계산된 95% 신뢰구간(대략 하한 ~6.9점대, 상한 ~7.5점대 부근, 실제 실행 결과값 기준으로 조정 필요)을 바탕으로 다음과 같은 전략을 세울 수 있어요.

1. 목표 기준선과의 비교를 통한 의사결정

만약 회사의 만족도 목표치가 "7.0점 이상 유지"라면, 신뢰구간의 하한값이 7.0을 넘는지가 핵심이에요.

하한값이 7.0 이상이라면 → "95% 확률로 실제 고객 만족도는 목표치 이상"이라고 자신 있게 보고할 수 있고, 현재 운영 방식을 유지하며 다른 영역(예: 신규 고객 확보)에 마케팅 자원을 집중할 수 있어요.
하한값이 7.0보다 낮다면 → 표본 평균은 7.0을 넘었더라도 "목표치를 밑돌 가능성을 배제할 수 없다"는 뜻이므로, 안심하지 말고 만족도 개선 캠페인(사후 서비스 강화, 불만 고객 인터뷰 등)을 선제적으로 추진해야 해요.

2. 신뢰구간 폭을 활용한 추가 조사 필요성 판단

신뢰구간의 폭이 넓다면(예: 1점 이상 차이), 표본 크기 40명으로는 만족도를 정밀하게 파악하기 어렵다는 뜻이에요. 이 경우 표본 크기를 100~200명 수준으로 늘려서 조사를 확대하는 걸 제안할 수 있어요. 정밀한 수치가 필요할수록(예: 신제품 출시 여부를 결정하는 중요한 지표라면) 표본을 키워 신뢰구간을 좁히는 투자가 정당화돼요.

3. 세그먼트별 비교 전략

전체 고객 대상 신뢰구간뿐 아니라, 연령대·구매 채널·지역 등으로 세분화한 그룹별 신뢰구간을 각각 구해서 비교해볼 수 있어요. 만약 특정 그룹의 신뢰구간이 전체 평균보다 확연히 낮다면(구간이 겹치지 않을 정도로), 그 그룹을 타겟으로 한 맞춤형 만족도 개선 캠페인을 우선순위로 설계할 수 있어요.

4. 시계열 추적을 통한 캠페인 효과 검증

마케팅 캠페인(예: 서비스 개선, 이벤트 진행) 전후로 각각 신뢰구간을 계산해서, 두 구간이 겹치지 않는다면 "캠페인이 통계적으로 유의미한 만족도 개선 효과를 냈다"고 판단할 근거로 삼을 수 있어요. 반대로 구간이 겹친다면, 눈에 보이는 평균 차이가 우연에 의한 것일 수 있으니 추가 캠페인이나 더 큰 표본으로 재검증이 필요해요.

요약하면, 신뢰구간은 단순히 "평균이 몇 점이다"라는 점 추정치보다 훨씬 풍부한 정보를 제공해요. 마케팅 전략을 세울 때 이 구간의 하한값(보수적 시나리오), 폭(추정의 정밀도), 그리고 다른 그룹/시점과의 겹침 여부를 종합적으로 활용하면 "숫자 하나"가 아니라 "불확실성을 고려한 의사결정"을 할 수 있게 돼요.

**🧠 데이터를 어떻게 읽을까요?**

- 신뢰구간은 몇 점에서 몇 점 사이인가요?

- 이 구간은 전체 모집단 평균을 포함하고 있나요?

- 이 결과를 바탕으로 고객 만족도가 충분히 높다고 말할 수 있을까요?

- 만약 신뢰구간이 너무 넓게 나왔다면, 그 이유는 무엇이고 어떻게 개선할 수 있을까요?

In [ ]:
# [문제 5] 데이터를 어떻게 읽을까요?
# 여기에 의견을 작성해주세요.
1. 신뢰구간은 몇 점에서 몇 점 사이인가요?

앞서 코드로 계산한 결과를 실제로 실행해보면, 표본 평균은 약 7.40점이고, 95% 신뢰구간은 대략 (7.05, 7.75) 사이로 나와요. 즉 "이 표본을 기준으로 봤을 때, 95% 확률로 전체 고객의 진짜 평균 만족도는 7.05점에서 7.75점 사이에 있다"고 해석할 수 있어요.

2. 이 구간은 전체 모집단 평균을 포함하고 있나요?

네, 포함하고 있어요. 모집단(10,000명) 전체의 실제 평균은 약 7.19점인데, 이 값은 신뢰구간 (7.05, 7.75) 안에 들어가 있어요. 이건 이번 표본이 "운 좋게" 모집단 평균을 잘 포착한 경우예요. (참고로 신뢰구간의 의미는 "이 특정 구간이 100번 중 95번 확률로 맞다"는 게 아니라, "이런 방식으로 신뢰구간을 반복해서 만들면 그중 95%가 실제 모평균을 포함한다"는 뜻이에요. 이번엔 그 95%에 해당하는 경우였던 거죠.)

3. 이 결과를 바탕으로 고객 만족도가 충분히 높다고 말할 수 있을까요?

이건 "충분히 높다"의 기준을 어디에 두느냐에 따라 달라져요.

만약 만족도 목표치가 예를 들어 7.0점이라면, 신뢰구간의 하한값(7.05)이 이미 7.0을 넘고 있으므로 "95% 신뢰 수준에서 목표치를 달성했다"고 어느 정도 자신 있게 말할 수 있어요.
하지만 목표치가 7.5점처럼 더 높게 설정되어 있다면, 신뢰구간 (7.05~7.75)이 7.5를 걸치고 있어서 "목표를 달성했다고 확신하기는 어렵다"고 봐야 해요 — 실제 평균이 7.05일 가능성도 배제할 수 없으니까요.

단순히 표본 평균(7.40)만 보고 "충분히 높다"고 단정하는 것보다, 신뢰구간의 하한값까지 고려해서 "최소한 이 정도는 보장된다"는 보수적인 관점으로 판단하는 게 더 안전한 접근이에요.

4. 만약 신뢰구간이 너무 넓게 나왔다면, 그 이유는 무엇이고 어떻게 개선할 수 있을까요?

신뢰구간이 넓어지는 원인은 크게 두 가지예요.

표본 크기(n)가 작을 때: SE = s/√n 공식에서 n이 작으면 SE가 커지고, 그만큼 margin of error(t_critical × SE)도 커져서 구간이 넓어져요. 이번 예제도 n=40이라는 비교적 작은 표본을 썼기 때문에 구간 폭이 0.7점 정도로 꽤 넓게 나온 편이에요.
표본의 변동성(표준편차)이 클 때: 응답자들의 만족도 점수가 들쭉날쭉하고 흩어져 있을수록(표준편차가 클수록) 표본 평균의 불확실성도 커져서 구간이 넓어져요.

개선 방법은:

표본 크기를 늘리기: 가장 확실한 방법이에요. 예를 들어 n을 40 → 160으로 4배 늘리면, SE는 이론상 절반으로 줄어들어서 신뢰구간 폭도 대략 절반으로 좁아져요.
측정 방식을 개선해 변동성 줄이기: 설문 문항을 더 명확하게 설계하거나, 극단적으로 편차가 큰 이상치 응답이 왜 나오는지 원인을 분석해서 데이터의 노이즈를 줄이는 것도 방법이에요.
신뢰수준을 조정하기: 다만 이건 "폭을 억지로 줄이는" 방법일 뿐 근본적인 해결책은 아니에요. 신뢰수준을 95%에서 90%로 낮추면 구간은 좁아지지만, 대신 "진짜 평균을 포함할 확률"도 함께 낮아지는 트레이드오프가 생기니 신중하게 접근해야 해요.